# Kafka Ingestion

This notebook connects to the Aiven Kafka cluster,
reads streaming OpenStack logs,
parses JSON messages,
and produces a structured streaming DataFrame.

Output:
stream_df

In [0]:
%run ./00_Project_Setup

# Log Guardian - Project Setup

This notebook contains all project-level configurations required by the Log Guardian streaming pipeline.

Responsibilities:
- Import required libraries
- Configure Kafka connection
- Configure Aiven authentication
- Load SSL certificate
- Define common project paths
- Define reusable variables

No data processing is performed in this notebook.

openstack-normal1,openstack-normal2,openstack-abnormal


In [0]:
# Read the Aiven CA certificate (PEM format)
with open("/Volumes/log-analytics/bronze/key_volume/ca.pem", "r") as f:
    ca_cert = f.read()

jaas = (f'kafkashaded.org.apache.kafka.common.security.plain.PlainLoginModule required '
        f'username="{AIVEN_USERNAME}" password="{AIVEN_PASSWORD}";')

df = (
    spark.read.format("kafka")
      .option("kafka.bootstrap.servers", AIVEN_BOOTSTRAP)
      .option("subscribe", KAFKA_TOPICS)
      .option("kafka.security.protocol", "SASL_SSL")
      .option("kafka.sasl.mechanism", "PLAIN")
      .option("kafka.sasl.jaas.config", jaas)
      .option("kafka.ssl.truststore.type", "PEM")               # ✅ use PEM, not JKS
      .option("kafka.ssl.truststore.certificates", ca_cert)     # ✅ pass cert content directly
      .option("kafka.ssl.endpoint.identification.algorithm", "https")
      .option("startingOffsets", "earliest")
      .load()
    )

df.show(5)

+--------------------+--------------------+-----------------+---------+------+--------------------+-------------+
|                 key|               value|            topic|partition|offset|           timestamp|timestampType|
+--------------------+--------------------+-----------------+---------+------+--------------------+-------------+
|[64 61 74 61 2F 6...|[7B 22 65 76 65 6...|openstack-normal1|        1|     0|2026-07-19 13:33:...|            0|
|[64 61 74 61 2F 6...|[7B 22 65 76 65 6...|openstack-normal1|        1|     1|2026-07-19 13:33:...|            0|
|[64 61 74 61 2F 6...|[7B 22 65 76 65 6...|openstack-normal1|        1|     2|2026-07-19 13:33:...|            0|
|[64 61 74 61 2F 6...|[7B 22 65 76 65 6...|openstack-normal1|        1|     3|2026-07-19 13:33:...|            0|
|[64 61 74 61 2F 6...|[7B 22 65 76 65 6...|openstack-normal1|        1|     4|2026-07-19 13:33:...|            0|
+--------------------+--------------------+-----------------+---------+------+----------

In [0]:
from pyspark.sql import functions as F
from pyspark.sql.types import *

# Schema matching log_parser.py output exactly
OPENSTACK_SCHEMA = StructType([
    StructField("event_id",       StringType(),  True),
    StructField("timestamp",      StringType(),  True),
    StructField("dataset_source", StringType(),  True),
    StructField("log_file",       StringType(),  True),
    StructField("service",        StringType(),  True),
    StructField("process_id",     StringType(),  True),
    StructField("log_level",      StringType(),  True),
    StructField("request_id",     StringType(),  True),
    StructField("user_id",        StringType(),  True),
    StructField("project_id",     StringType(),  True),
    StructField("instance_id",    StringType(),  True),
    StructField("client_ip",      StringType(),  True),
    StructField("http_method",    StringType(),  True),
    StructField("http_path",      StringType(),  True),
    StructField("message",        StringType(),  True),
    StructField("status_code",    IntegerType(), True),
    StructField("response_time",  DoubleType(),  True),
])

# Parse: binary → string → JSON → columns
parsed_df = (
    df
    .withColumn("json_str", F.col("value").cast("string"))
    .withColumn("parsed", F.from_json(F.col("json_str"), OPENSTACK_SCHEMA))
    .select(
        "parsed.*",

        # Kafka Metadata
        F.col("topic").alias("kafka_topic"),
        F.col("partition").alias("kafka_partition"),
        F.col("offset").alias("kafka_offset"),
        F.col("timestamp").alias("kafka_timestamp"),

        # Ingestion Metadata
        F.current_timestamp().alias("ingestion_timestamp"),
    )
)

parsed_df.show(5, truncate=False)


+------------------------------------+-----------------------+--------------+------------+------------------------------+----------+---------+------------------------------------+--------------------------------+--------------------------------+-----------+----------+-----------+---------------------------------------------------+-------------------------------------------------------+-----------+-------------+-----------------+---------------+------------+-----------------------+-------------------------+
|event_id                            |timestamp              |dataset_source|log_file    |service                       |process_id|log_level|request_id                          |user_id                         |project_id                      |instance_id|client_ip |http_method|http_path                                          |message                                                |status_code|response_time|kafka_topic      |kafka_partition|kafka_offset|kafka_timestamp        |ing

In [0]:
# Re-read as a STREAM (not batch)
stream_df = (spark.readStream.format("kafka")
    .option("kafka.bootstrap.servers", AIVEN_BOOTSTRAP)
    .option("subscribe", KAFKA_TOPICS)
    .option("kafka.security.protocol", "SASL_SSL")
    .option("kafka.sasl.mechanism", "PLAIN")
    .option("kafka.sasl.jaas.config", jaas)
    .option("kafka.ssl.truststore.type", "PEM")
    .option("kafka.ssl.truststore.certificates", ca_cert)
    .option("kafka.ssl.endpoint.identification.algorithm", "https")
    .option("startingOffsets", "earliest")
    .load()
    .withColumn("json_str", F.col("value").cast("string"))
    .withColumn("parsed",   F.from_json(F.col("json_str"), OPENSTACK_SCHEMA))
    .select(
    "parsed.*",
    F.col("topic").alias("kafka_topic"),
    F.col("partition").alias("kafka_partition"),
    F.col("offset").alias("kafka_offset"),
    F.col("timestamp").alias("kafka_timestamp"),
    F.current_timestamp().alias("ingestion_timestamp"),
    )
)

# Write stream → Delta Lake Bronze table
bronze_query = (
    stream_df.writeStream
        .format("delta")
        .outputMode("append")
        .partitionBy("log_file")

        # NEW CHECKPOINT
        .option(
            "checkpointLocation",
            "/Volumes/log-analytics/bronze/checkpoint_volume/checkpoints/bronze_stream_v2"
        )

        .trigger(availableNow=True)
        .start("/Volumes/log-analytics/bronze/key_volume/bronze_delta_v2")
)

print(f"✅ Stream started! Query ID: {bronze_query.id}")
print("⏳ Wait 30 seconds then run the next cell...")


✅ Stream started! Query ID: 560a6a8b-ef75-404c-85f8-17d536e3403c
⏳ Wait 30 seconds then run the next cell...


In [0]:
bronze_df = spark.read.format("delta").load("/Volumes/log-analytics/bronze/key_volume/bronze_delta_v2")
print(f"Total rows: {bronze_df.count()}")
bronze_df.groupBy("log_level").count().orderBy("count", ascending=False).show()
bronze_df.groupBy("log_file").count().show()


Total rows: 281900
+---------+------+
|log_level| count|
+---------+------+
|     INFO|277119|
|  WARNING|  4515|
|    ERROR|   265|
| CRITICAL|     1|
+---------+------+

+------------------+------+
|          log_file| count|
+------------------+------+
|nova-scheduler.log|  1020|
|  nova-compute.log|131643|
|      nova-api.log|149237|
+------------------+------+

